In [7]:
!pip install -q transformers datasets peft bitsandbytes accelerate pandas scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


In [10]:
import pandas as pd

SUBJECTS = ["Math", "English", "Science", "History"]

def create_training_data(mid_path, end_path):
    mid = pd.read_csv(mid_path)
    end = pd.read_csv(end_path)

    df = pd.merge(mid, end, on=["StudentID", "Name"], how="outer")

    data = []

    for _, row in df.iterrows():

        text = f"StudentID: {row['StudentID']}\nName: {row['Name']}\n\n"

        summary = ""
        total, count = 0, 0

        for sub in SUBJECTS:
            m = row.get(f"{sub}_Mid")
            e = row.get(f"{sub}_End")

            text += f"{sub}: {m} → {e}\n"

            if pd.notna(m) and pd.notna(e):
                diff = float(e) - float(m)
                total += diff
                count += 1

                if diff > 0:
                    summary += f"{sub}: Improved\n"
                else:
                    summary += f"{sub}: Declined\n"
            else:
                summary += f"{sub}: Missing\n"

        avg = total / count if count > 0 else 0
        status = "Improved" if avg > 0 else "Declined"

        output = f"""
Performance Summary:
{summary}

Final Status: {status}
Verdict: Student shows mixed performance across subjects.
"""

        full_text = text + "\n" + output
        data.append(full_text)

    return data

In [11]:
train_data = create_training_data("mid.csv", "end.csv")

In [12]:
print(train_data[0])

StudentID: 701
Name: Aanya

Math: nan → nan
English: 55.0 → 99.0
Science: 70.0 → 86.0
History: 65.0 → 53.0


Performance Summary:
Math: Missing
English: Improved
Science: Improved
History: Declined


Final Status: Improved
Verdict: Student shows mixed performance across subjects.



In [13]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-4)

model.train()

for epoch in range(1):   # keep 1 for now (safe on T4)
    for i, sample in enumerate(train_data):

        inputs = tokenizer(
            sample,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to("cuda")

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=inputs["input_ids"]
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if i % 1 == 0:
            print(f"Step {i}, Loss: {loss.item()}")

Step 0, Loss: 2.3489439487457275
Step 1, Loss: 2.3157880306243896
Step 2, Loss: 1.980029821395874
Step 3, Loss: 1.9232966899871826
Step 4, Loss: 2.344320297241211
Step 5, Loss: 1.839004397392273
Step 6, Loss: 2.1362533569335938
Step 7, Loss: 2.179422616958618
Step 8, Loss: 2.1254162788391113
Step 9, Loss: 1.6467357873916626
Step 10, Loss: 1.9772177934646606
Step 11, Loss: 1.8272879123687744
Step 12, Loss: 1.9371895790100098
Step 13, Loss: 1.888831377029419
Step 14, Loss: 1.6846927404403687
Step 15, Loss: 1.4320873022079468
Step 16, Loss: 1.3710485696792603
Step 17, Loss: 1.7064746618270874
Step 18, Loss: 1.7156157493591309
Step 19, Loss: 1.3709156513214111
Step 20, Loss: 1.3963541984558105
Step 21, Loss: 1.4492881298065186
Step 22, Loss: 1.3641207218170166
Step 23, Loss: 1.30055832862854
Step 24, Loss: 1.0707205533981323
Step 25, Loss: 1.0468312501907349
Step 26, Loss: 1.032894492149353
Step 27, Loss: 0.8837176561355591
Step 28, Loss: 1.0331131219863892
Step 29, Loss: 0.995755195617675

In [14]:
model.save_pretrained("qwen_csv_model")
tokenizer.save_pretrained("qwen_csv_model")

('qwen_csv_model/tokenizer_config.json',
 'qwen_csv_model/chat_template.jinja',
 'qwen_csv_model/tokenizer.json')

# **TESTING**

In [15]:
test_text = """
StudentID: 701
Name: Esha

Math: 71 → 75
English: NA → 80
Science: 72 → 78
History: 57 → 60

Give performance summary and verdict.
"""

inputs = tokenizer(test_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120)

print(tokenizer.decode(out[0], skip_special_tokens=True))


StudentID: 701
Name: Esha

Math: 71 → 75
English: NA → 80
Science: 72 → 78
History: 57 → 60

Give performance summary and verdict.
Performance Summary:
Math: Improved
English: Missing
Science: Improved
History: Improved

Verdict: Student shows mixed performance across subjects. Overall, slight improvements observed in Math, Science, and History. English performance is missing, which could indicate a gap or challenge area. Verdict leans towards mixed performance but with positive trends in key areas.
"""

# Extracting performance details for each subject
math_improved = "Improved" if math_improvement else "Missing"
english_missing = "Missing" if english_performance == "NA" else "Improved"
science_improved = "Improved"


In [16]:
import pandas as pd

mid = pd.read_csv("mid1.csv")
end = pd.read_csv("end1.csv")

df = pd.merge(mid, end, on=["StudentID", "Name"], how="outer")

In [17]:
SUBJECTS = ["Math", "English", "Science", "History"]

def clean(x):
    if pd.isna(x) or x == "NA":
        return None
    return float(x)

def compute(row):
    result = {}

    for sub in SUBJECTS:
        m = clean(row.get(f"{sub}_Mid"))
        e = clean(row.get(f"{sub}_End"))

        if m is None or e is None:
            diff = "Missing"
            status = "Missing"
        else:
            diff = e - m
            status = "Improved" if diff > 0 else "Declined"

        result[f"{sub}_Diff"] = diff
        result[f"{sub}_Status"] = status

    return result

# **LOADING BASE MODEL + LORA WEIGHTS**

In [19]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

base = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

model = PeftModel.from_pretrained(base, "qwen_csv_model")
model.eval()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear

In [20]:
SUBJECTS = ["Math", "English", "Science", "History"]

def clean(x):
    if pd.isna(x) or x == "NA":
        return None
    return float(x)

def compute(row):
    result = {}

    for sub in SUBJECTS:
        m = clean(row.get(f"{sub}_Mid"))
        e = clean(row.get(f"{sub}_End"))

        if m is None or e is None:
            diff = "Missing"
            status = "Missing"
        else:
            diff = e - m
            status = "Improved" if diff > 0 else "Declined"

        result[f"{sub}_Diff"] = diff
        result[f"{sub}_Status"] = status

    return result

In [21]:
def get_verdict(row, diff_data):
    prompt = f"""
Student: {row['Name']}

Performance:
"""

    for sub in SUBJECTS:
        prompt += f"{sub}: {diff_data[sub+'_Status']}\n"

    prompt += "\nGive a short final verdict (1-2 lines)."

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.3
        )

    text = tokenizer.decode(out[0], skip_special_tokens=True)

    return text.split("verdict")[-1]

In [22]:
def run_demo(mid_file, end_file):
    mid = pd.read_csv(mid_file)
    end = pd.read_csv(end_file)

    df = pd.merge(mid, end, on=["StudentID", "Name"], how="outer")

    results = []

    for _, row in df.iterrows():
        diff_data = compute(row)

        verdict = get_verdict(row, diff_data)

        row_data = {
            "StudentID": row["StudentID"],
            "Name": row["Name"]
        }

        row_data.update(diff_data)
        row_data["Verdict"] = verdict

        results.append(row_data)

    final_df = pd.DataFrame(results)
    final_df.to_csv("final_output.csv", index=False)

    return final_df

In [23]:
output = run_demo("mid1.csv", "end1.csv")
output

,StudentID,Name,Math_Diff,Math_Status,English_Diff,English_Status,Science_Diff,Science_Status,History_Diff,History_Status,Verdict
0,701,Aanya,Missing,Missing,44.0,Improved,16.0,Improved,-12.0,Declined,(1-2 lines). Teacher: Aanya shows mixed perfo...
1,702,Krishna,Missing,Missing,22.0,Improved,-36.0,Declined,-21.0,Declined,(1-2 lines). Teacher: Krishna shows improveme...
2,703,Sneha,6.0,Improved,-33.0,Declined,13.0,Improved,3.0,Improved,.)
3,704,Riaan,-10.0,Declined,15.0,Improved,11.0,Improved,-31.0,Declined,(1-2 lines). Teacher: Riaan shows mixed perfo...
4,705,Saisha,8.0,Improved,Missing,Missing,Missing,Missing,6.0,Improved,(1-2 lines). Expert: Student Saisha shows mix...
...,...,...,...,...,...,...,...,...,...,...,...
95,796,Ansh,2.0,Improved,-4.0,Declined,-20.0,Declined,-23.0,Declined,(1-2 lines). Assistant: Ansh shows mixed perf...
96,797,Aarav,Missing,Missing,-34.0,Declined,-23.0,Declined,47.0,Improved,(1-2 lines). Assistant: Aarav shows improveme...
97,798,Nikhil,-47.0,Declined,13.0,Improved,4.0,Improved,42.0,Improved,: Nikhil demonstrates steady progress in his a...
98,799,Raghav,-31.0,Declined,Missing,Missing,Missing,Missing,42.0,Improved,(1-2 lines). Expert: Based on the provided pe...
